In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pickle

from src.data_utils import (
    load_csv_plan,
    load_tabular_features,
    make_class_mappings,
    load_logmel_features,
)
from sklearn.ensemble import RandomForestClassifier

from src.models import svm_model, random_forest_model
from src.evaluation import run_predefined_fold_cv, summarize_fold_metrics
from src.cnn import train_cnn_cv

import torch
import optuna

ROOT = Path.cwd().parent.resolve()
DATA_DIR = ROOT / "data"

FEATURE_PATH = DATA_DIR / "features/tabular_audio_features.npz"
LOGMEL_PATH = DATA_DIR / "features/logmel_spectogram.npz"
CV_PLAN_PATH = DATA_DIR / "processed/cv_plan_10fold.json"
OUTPUT_DIR = ROOT / "results/models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Loading Processed Data

In [5]:
data = load_tabular_features(FEATURE_PATH)
cv_plan = load_csv_plan(CV_PLAN_PATH)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
labels = data["labels"]
folds = data["folds"].astype(np.int64)
filenames = data["filenames"]
feature_names = data["feature_names"]

print("X:", X.shape)
print("Y:", y.shape)
print("folds:", folds.shape)
print("CV folds:", len(cv_plan))

class_table, class_ids, class_names, id_to_class = make_class_mappings(y, labels)
display(class_table)

num_classes = len(class_ids)

X: (8732, 176)
Y: (8732,)
folds: (8732,)
CV folds: 10


,class_id,class
0,0,air_conditioner
1,1,car_horn
2,2,children_playing
3,3,dog_bark
4,4,drilling
5,5,engine_idling
6,6,gun_shot
7,7,jackhammer
8,8,siren
9,9,street_music


In [6]:
data_logmel = load_logmel_features(LOGMEL_PATH)

X_logmel = data_logmel["X"].astype(np.float32)
y_logmel = data_logmel["y"].astype(np.int64)
labels_logmel = data_logmel["labels"]
folds_logmel = data_logmel["folds"].astype(np.int64)
filenames_logmel = data_logmel["filenames"]

print("X:", X_logmel.shape)
print("Y:", y_logmel.shape)
print("folds:", folds_logmel.shape)

X: (8732, 128, 173)
Y: (8732,)
folds: (8732,)


## Part 1: SVM Tuning

In [8]:
svm_experiments = [
    {"name": "baseline", "C": 10, "gamma": "scale"},
    {"name": "low_C", "C": 1, "gamma": "scale"},
    {"name": "high_C", "C": 100, "gamma": "scale"},
    {"name": "gamma_001", "C": 10, "gamma": 0.01},
    {"name": "gamma_0001", "C": 10, "gamma": 0.001},
]

results = []

best_score = -1
best_fold_metrics = None
best_predictions = None
best_summary = None
best_experiment = None

for experiment in svm_experiments:
    print("Experiment Config:")
    print(experiment, end="\n\n")
    fold_metrics, predictions_df = run_predefined_fold_cv(
        model_fn=lambda: svm_model(
            seed=42, C=experiment["C"], gamma=experiment["gamma"]
        ),
        X=X,
        y=y,
        folds=folds,
        filenames=filenames,
        cv_plan=cv_plan,
        id_to_class=id_to_class,
    )

    summary = summarize_fold_metrics(fold_metrics)

    results.append({"experiment": experiment["name"], **summary})
    print("\n\n")

    if summary["f1_macro_mean"] > best_score:
        best_score = summary["f1_macro_mean"]
        best_fold_metrics = fold_metrics
        best_predictions = predictions_df
        best_summary = summary
        best_experiment = experiment

svm_results = pd.DataFrame(results)

Experiment Config:
{'name': 'baseline', 'C': 10, 'gamma': 'scale'}

Fold 1: accuracy=0.6231, macro_f1=0.6426, weighted_f1=0.5947
Fold 2: accuracy=0.6520, macro_f1=0.6627, weighted_f1=0.6411
Fold 3: accuracy=0.5849, macro_f1=0.6045, weighted_f1=0.5691
Fold 4: accuracy=0.6788, macro_f1=0.6672, weighted_f1=0.6680
Fold 5: accuracy=0.8066, macro_f1=0.8156, weighted_f1=0.8044
Fold 6: accuracy=0.6173, macro_f1=0.6491, weighted_f1=0.6156
Fold 7: accuracy=0.7303, macro_f1=0.7380, weighted_f1=0.7254
Fold 8: accuracy=0.6489, macro_f1=0.6737, weighted_f1=0.6412
Fold 9: accuracy=0.7267, macro_f1=0.7468, weighted_f1=0.7176
Fold 10: accuracy=0.7240, macro_f1=0.7405, weighted_f1=0.7204



Experiment Config:
{'name': 'low_C', 'C': 1, 'gamma': 'scale'}

Fold 1: accuracy=0.6369, macro_f1=0.6653, weighted_f1=0.6167
Fold 2: accuracy=0.6655, macro_f1=0.6817, weighted_f1=0.6579
Fold 3: accuracy=0.5686, macro_f1=0.5863, weighted_f1=0.5580
Fold 4: accuracy=0.6828, macro_f1=0.6703, weighted_f1=0.6790
Fold 5: ac

In [10]:
svm_output = OUTPUT_DIR / "svm"
svm_output.mkdir(parents=True, exist_ok=True)

svm_results.to_csv(svm_output / "tuning_results.csv", index=False)

best_fold_metrics.to_csv(svm_output / "fold_metrics.csv", index=False)
best_predictions.to_csv(svm_output / "predictions.csv", index=False)

with open(svm_output / "summary_metrics.json", "w") as f:
    json.dump(best_summary, f, indent=4)

with open(svm_output / "best_config.json", "w") as f:
    json.dump(best_experiment, f, indent=4)

## Part 2: Random Forest

In [11]:
rf_experiments = [
    {"name": "baseline", "n_estimators": 300, "max_depth": None, "min_sample_leaf": 1},
    {
        "name": "more_trees",
        "n_estimators": 500,
        "max_depth": None,
        "min_sample_leaf": 1,
    },
    {
        "name": "limited_depth",
        "n_estimators": 300,
        "max_depth": 30,
        "min_sample_leaf": 1,
    },
    {
        "name": "regularized_leaf2",
        "n_estimators": 300,
        "max_depth": None,
        "min_sample_leaf": 2,
    },
    {
        "name": "regularized_leaf4",
        "n_estimators": 300,
        "max_depth": None,
        "min_sample_leaf": 4,
    },
]

results = []

best_score = -1
best_fold_metrics = None
best_predictions = None
best_summary = None
best_experiment = None

for experiment in rf_experiments:
    print("Experiment Config:")
    print(experiment, end="\n\n")
    fold_metrics, predictions_df = run_predefined_fold_cv(
        model_fn=lambda: random_forest_model(
            seed=42,
            n_estimators=experiment["n_estimators"],
            max_depth=experiment["max_depth"],
            min_samples_leaf=experiment["min_sample_leaf"],
        ),
        X=X,
        y=y,
        folds=folds,
        filenames=filenames,
        cv_plan=cv_plan,
        id_to_class=id_to_class,
    )

    summary = summarize_fold_metrics(fold_metrics)

    results.append({"experiment": experiment["name"], **summary})
    print("\n\n")

    if summary["f1_macro_mean"] > best_score:
        best_score = summary["f1_macro_mean"]
        best_fold_metrics = fold_metrics
        best_predictions = predictions_df
        best_summary = summary
        best_experiment = experiment

rf_results = pd.DataFrame(results)

Experiment Config:
{'name': 'baseline', 'n_estimators': 300, 'max_depth': None, 'min_sample_leaf': 1}

Fold 1: accuracy=0.6586, macro_f1=0.6680, weighted_f1=0.6525
Fold 2: accuracy=0.6610, macro_f1=0.6918, weighted_f1=0.6544
Fold 3: accuracy=0.5892, macro_f1=0.6241, weighted_f1=0.5795
Fold 4: accuracy=0.6838, macro_f1=0.6667, weighted_f1=0.6771
Fold 5: accuracy=0.7318, macro_f1=0.7399, weighted_f1=0.7281
Fold 6: accuracy=0.6561, macro_f1=0.6770, weighted_f1=0.6534
Fold 7: accuracy=0.6551, macro_f1=0.6634, weighted_f1=0.6500
Fold 8: accuracy=0.6253, macro_f1=0.6490, weighted_f1=0.6184
Fold 9: accuracy=0.6838, macro_f1=0.6963, weighted_f1=0.6757
Fold 10: accuracy=0.6989, macro_f1=0.7168, weighted_f1=0.6987



Experiment Config:
{'name': 'more_trees', 'n_estimators': 500, 'max_depth': None, 'min_sample_leaf': 1}

Fold 1: accuracy=0.6575, macro_f1=0.6667, weighted_f1=0.6516
Fold 2: accuracy=0.6554, macro_f1=0.6865, weighted_f1=0.6497
Fold 3: accuracy=0.5816, macro_f1=0.6185, weighted_f1=0.

In [12]:
rf_final = random_forest_model(
    seed=42,
    n_estimators=best_experiment["n_estimators"],
    max_depth=best_experiment["max_depth"],
    min_samples_leaf=best_experiment["min_sample_leaf"],
)
rf_final.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [13]:
importance = rf_final.feature_importances_

importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": importance}
).sort_values("importance", ascending=False)

In [14]:
rf_output = OUTPUT_DIR / "random_forest"
rf_output.mkdir(parents=True, exist_ok=True)

rf_results.to_csv(rf_output / "tuning_results.csv", index=False)
importance_df.to_csv(rf_output / "rf_feature_importance.csv", index=False)

best_fold_metrics.to_csv(rf_output / "fold_metrics.csv", index=False)
best_predictions.to_csv(rf_output / "predictions.csv", index=False)

with open(rf_output / "summary_metrics.json", "w") as f:
    json.dump(best_summary, f, indent=4)

with open(rf_output / "best_config.json", "w") as f:
    json.dump(best_experiment, f, indent=4)

## Part 3: CNN

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


def objective(trial):
    config = {
        "seed": 42,
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "max_epoch": 12,
        "patience": 3,
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.2, 0.6),
        "validation_size": 0.2,
        "num_workers": 2,
    }

    fold_metrics_df, _, _, _, _ = train_cnn_cv(
        X_logmel,
        y_logmel,
        folds_logmel,
        filenames_logmel,
        cv_plan,
        id_to_class,
        num_classes,
        device,
        config,
        verbose=False,
        test_folds=[1, 5, 9],
    )

    return fold_metrics_df["f1_macro"].mean()

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("Best value:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
best_cnn_config = {
    "seed": 42,
    "max_epoch": 40,
    "patience": 7,
    "validation_size": 0.2,
    "num_workers": 2,
    **study.best_params,
}

In [ ]:
fold_metrics_df, predictions_df, histories, best_state, best_info = train_cnn_cv(
    X=X_logmel,
    y=y_logmel,
    folds=folds_logmel,
    filenames=filenames_logmel,
    cv_plan=cv_plan,
    id_to_class=id_to_class,
    num_classes=num_classes,
    device=device,
    config=best_cnn_config,
    verbose=True,
)

In [ ]:
summary_metrics = summarize_fold_metrics(fold_metrics_df)

out_path = OUTPUT_DIR / "cnn"
out_path.mkdir(parents=True, exist_ok=True)

fold_metrics_df.to_csv(out_path / "fold_metrics.csv", index=False)
predictions_df.to_csv(out_path / "predictions.csv", index=False)

with open(out_path / "summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

with open(out_path / "best_config.json", "w") as f:
    json.dump(best_cnn_config, f, indent=4)

with open(out_path / "histories.pk1", "wb") as f:
    pickle.dump(histories, f)

torch.save(best_state, out_path / "best_model.pt")

optuna_results = study.trials_dataframe()

optuna_results.to_csv(out_path / "optuna_results.csv", index=False)